# Tarea grupal: "EnviRápido", centro de distribución de paquetes


**Grupos de 5 · Lenguaje: Python 

Este cuaderno contiene el **enunciado**, los **mínimos necesarios**, los **puntos de libertad**, los **entregables** y la **rúbrica**. Al final hay un espacio de trabajo con una celda de código por cada mínimo.



---
# 1. El problema
Una empresa de mensajería, **EnviRápido**, tiene un centro de distribución que hoy funciona con cuadernos y hojas de papel. Su dueño sufre estos problemas:

- Cada día llegan cientos de paquetes y se atienden **en el orden en que llegaron**, pero a veces los empleados se saltan paquetes sin darse cuenta.
- Los clientes llaman preguntando *"¿dónde está mi paquete?"* y el empleado tarda minutos en encontrarlo revisando hoja por hoja.
- A veces un empleado marca un paquete como "entregado" por error y no hay forma de corregirlo ordenadamente.
- La empresa reparte a **zonas/sucursales** (cientos de ellas, con un código numérico) y necesita saber rápidamente a cuál corresponde cada código.
- Hay paquetes que llegan **sin código de seguimiento**, solo con el nombre del cliente, y también hay que poder ubicarlos.

**Tu grupo es una empresa de software** que debe ofrecerle al dueño un prototipo que resuelva estos problemas. Ustedes deciden qué datos tiene un paquete, cómo se organiza el sistema y qué estructuras usar. El dueño no sabe de programación: tendrán que **convencerlo** de que su solución es buena y de por qué.

---
# 2. Mínimos necesarios (para aprobar)

1. **Modelado propio:** definir qué es un paquete, un cliente y una zona (clases o estructuras), y explicar por escrito qué decidieron y por qué.
2. **Recepción y atención en orden de llegada**, con posibilidad de ver quién sigue.
3. **Consulta rápida por código de seguimiento**, usando **una tabla hash implementada por ustedes** (sin usar `dict` ni `set` de Python para la estructura principal), con **manejo de colisiones**.
4. **Corrección de errores** (deshacer acciones, al menos las últimas) usando una estructura adecuada.
5. **Búsqueda de zona por código** sobre una lista ordenada, usando búsqueda binaria.
6. **Búsqueda de paquetes por nombre de cliente** (sin código), usando búsqueda secuencial.
7. **Pruebas:** un programa de ejemplo con al menos 20 paquetes que demuestre cada funcionalidad, incluyendo casos límite (cola vacía, código inexistente, colisión, deshacer sin historial).
8. **Medición:** comparar cuántas comparaciones hace cada búsqueda (secuencial vs. binaria vs. hash) con 100, 1000 y 10000 datos, y mostrar los resultados.

---
# 3. Puntos de libertad (donde tienen que pensar)
Estas decisiones **no las da el enunciado**; deben tomarlas y justificarlas:

- ¿Qué pasa con los paquetes urgentes o con prioridad? ¿Rompen la regla del orden de llegada?
- ¿Qué se hace si dos paquetes tienen el mismo nombre de cliente?
- ¿Cuál es el tamaño de la tabla hash y qué función hash usan? ¿Qué hacen cuando se llena?
- ¿Qué acciones se pueden deshacer y cuáles no?
- ¿La lista de zonas puede cambiar? Si cambia, ¿cómo se mantiene ordenada?
- ¿Cómo reportan al usuario que algo no se encontró?

**Extras opcionales (para nota adicional):** reasignar paquetes entre zonas, estadísticas del día, interfaz de menú por consola, carga de datos desde archivo.

---
# 4. Entregables y "venta"

| Entregable | Detalle |
|---|---|
| Código | Este cuaderno `.ipynb` (o archivos `.py`), comentado y ordenado |
| **Presentación de venta (10 min )** | Demostración en vivo + argumento de por qué su solución es la mejor |

**Reglas de la "venta":** los 5 integrantes deben hablar. El público (el resto de la clase y el docente) actúa como el dueño de la empresa y puede hacer preguntas como *"¿qué pasa si hay 1 millón de paquetes?"* o *"¿por qué no usaron una lista para esto?"*.



---
# ESPACIO DE TRABAJO DEL GRUPO

### Equipo y roles
Completen esta tabla (haciendo doble clic en la celda). Los roles deben **rotar** al preparar la defensa.

| Integrante | Rol principal | Rol de la defensa |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

### Registro de decisiones de diseño
Escriban aquí, en pocas líneas, qué decidieron para cada punto de libertad y **por qué** (esto alimenta el informe).

- **Paquetes urgentes:** 
- **Nombres de cliente repetidos:** 
- **Tamaño y función hash / tabla llena:** 
- **Qué se puede deshacer y qué no:** 
- **Zonas (¿cambian? ¿cómo se mantienen ordenadas?):** 
- **Cómo se informa que algo no se encontró:** 

### Preparación: módulos
Importen aquí lo que necesiten (por ejemplo `randint` para generar datos de prueba). **Recuerden:** no usar `dict` ni `set` para la tabla hash principal.

In [ ]:
from random import randint
import random

### Mínimo 1 · Modelado: paquete, cliente y zona
Definan las clases o estructuras. Antes de programar, escriban en papel los **sustantivos** y los **verbos** del problema.

In [ ]:
class Clientes:
    def __init__(self, nombre, apellido, telefono, llegada=0):
        self.nombre = nombre
        self.apellido = apellido
        self.telefono = telefono
        self.llegada = llegada
        self.espera = 0
    def __str__(self):
        return f"Cliente: {self.nombre} {self.apellido} Tel:({self.telefono})"

class Zona:
    def __init__(self, codigo_postal, nombre_zona):
        self.codigo_postal = codigo_postal
        self.nombre_zona = nombre_zona
    def __str__(self):
        return f"Zona: {self.codigo_postal} - {self.nombre_zona}"
    def __lt__(self, other):
        return self.codigo_postal < other.codigo_postal

class Paquete:
    def __init__(self, codigo_seguimiento, cliente, zona, description=""):
        self.codigo_seguimiento = codigo_seguimiento
        self.cliente = cliente
        self.zona = zona
        self.descripcion = description
        self.estado = "En depósito"
    def __str__(self):
        return f"Se registró el paquete con código de seguimiento: {self.codigo_seguimiento} Cliente: {self.cliente} Zona: {self.zona} Descripción: {self.descripcion}"

### Mínimo 2 · Recepción y atención en orden de llegada
Necesitan: recibir un paquete, atender el siguiente y **ver quién sigue** sin sacarlo.

In [ ]:
class Cola:
    def __init__(self):
        self.queue = []
    def is_empty(self):
        return len(self.queue) == 0
    def enqueue(self, item):
        self.queue.append(item)
    def dequeue(self):
        return None if self.is_empty() else self.queue.pop(0)
    def peek(self):
        return None if self.is_empty() else self.queue[0]
    def size(self):
        return len(self.queue)
    def __str__(self):
        return "Cola (frente--final):" + str([str(x) for x in self.queue])

class Pila:
    def __init__(self):
        self.stack = []
    def is_empty(self):
        return len(self.stack) == 0
    def push(self, item):
        self.stack.append(item)
    def pop(self):
        return None if self.is_empty() else self.stack.pop()
    def peek(self):
        return None if self.is_empty() else self.stack[-1]
    def size(self):
        return len(self.stack)
    def __str__(self):
        return "Pila (fondo-- tope):" + str([str(x) for x in self.stack])

class Paqueteria:
    def __init__(self):
        self.actual = None
        self.pausados = Pila()
        self.atendidos = []

    def llamar_siguiente(self, sala, minuto):
        cliente = sala.dequeue()
        if cliente is None:
            print("La sala de espera está vacía")
            return
        cliente.espera = minuto - cliente.llegada
        self.actual = cliente
        print(f"En el minuto {minuto} Atiende a: {cliente} espero {cliente.espera}")

    def terminar(self, minuto):
        if self.actual is None:
            return
        self.atendidos.append(self.actual)
        print(f"En el minuto {minuto} Termina la consulta de {self.actual}")
        self.actual = self.pausados.pop()
        if self.actual is not None:
            print(f"En el minuto {minuto} Retorna la consulta de {self.actual}")

### Mínimo 3 · Consulta por código de seguimiento (tabla hash propia)
Implementen la tabla hash completa: función hash, insertar, buscar y manejo de colisiones. **Sin `dict` ni `set`.**

In [ ]:
class HashTable:
    def __init__(self, size):
        self.size = size
        self.table = [[] for _ in range(size)]

    def _hash(self, key):
        return sum(map(ord, str(key))) % self.size

    def put(self, key, value):
        idx = self._hash(key)
        for item in self.table[idx]:
            if item[0] == key:
                item[1] = value
                return
        self.table[idx].append([key, value])

    def buscar_hash(self, key):
        idx = self._hash(key)
        comparaciones = 0
        for k, v in self.table[idx]:
            comparaciones += 1
            if k == key:
                return v, comparaciones
        return None, comparaciones

    def __str__(self):
        resultado = "Tabla hash\n"
        for i, bucket in enumerate(self.table):
            if bucket:
                resultado += f"Índice {i}: {bucket}\n"
        return resultado

def generar_codigo(tabla_hash):
    while True:
        numero_aleatorio = randint(100, 999)
        codigo_candidato = f"TRK-{numero_aleatorio}"
        resultado, _ = tabla_hash.buscar_hash(codigo_candidato)
        if resultado is None:
            return codigo_candidato

### Mínimo 4 · Deshacer acciones
Decidan qué acciones se registran y **qué información hay que guardar** para poder revertirlas.

In [ ]:
class HistorialDeshacer:
    def __init__(self):
        self.pila = []

    def registrar_accion(self, descripcion, paquete, estado_anterior):
        self.pila.append({
            'descripcion': descripcion,
            'paquete': paquete,
            'estado_anterior': estado_anterior
        })

    def deshacer(self):
        if len(self.pila) == 0:
            return "No hay acciones para deshacer en el historial."
        accion = self.pila.pop()
        paquete = accion['paquete']
        paquete.estado = accion['estado_anterior']
        return f"Acción deshecha: '{accion['descripcion']}'. Paquete {paquete.codigo_seguimiento} restaurado a estado: {paquete.estado}"

### Mínimo 5 · Búsqueda de zona por código (búsqueda binaria)
La lista de zonas debe estar **ordenada** antes de buscar.

In [ ]:
def busqueda_binaria_zona(lista_zonas, codigo_postal_buscado):
    inicio = 0
    fin = len(lista_zonas) - 1
    comparaciones = 0

    while inicio <= fin:
        comparaciones += 1
        medio = (inicio + fin) // 2
        if lista_zonas[medio].codigo_postal == codigo_postal_buscado:
            return lista_zonas[medio], comparaciones
        elif lista_zonas[medio].codigo_postal < codigo_postal_buscado:
            inicio = medio + 1
        else:
            fin = medio - 1
    return None, comparaciones

### Mínimo 6 · Búsqueda de paquetes por nombre de cliente (búsqueda secuencial)
Piensen qué devuelve su búsqueda si hay varios paquetes con el mismo nombre.

In [ ]:
def seq_search(lista, nombre_busc, apellido_busc):
    comparaciones = 0
    for i in range(len(lista)):
        comparaciones += 1
        cliente_actual = lista[i].cliente.lower()
        if nombre_busc.lower() in cliente_actual and apellido_busc.lower() in cliente_actual:
            return i, comparaciones
    return -1, comparaciones

def seq_search_todos(lista, nombre):
    indices = []
    comparaciones = 0
    for i in range(len(lista)):
        comparaciones += 1
        if lista[i].cliente.lower() == nombre.lower():
            indices.append(i)
    return indices, comparaciones

### Mínimo 7 · Pruebas con al menos 20 paquetes
Demuestren **cada** funcionalidad. Incluyan casos límite: cola vacía, código inexistente, colisión y deshacer sin historial.

In [ ]:
sala = Cola()
paquetes = Cola()
historial = HistorialDeshacer()
lista_zonas = [
    Zona("170110", "Sur"),
    Zona("170150", "Norte"),
    Zona("170200", "Centro"),
    Zona("170300", "Valle"),
]
lista_zonas.sort()
paqueteria = Paqueteria()
tabla_hash = HashTable(50)

nombres = ["Ana", "Beto", "Carlos", "Diana", "Eduardo", "Fernanda", "Gabriel", "Hilda", "Ivan", "Julia", 
           "Kevin", "Laura", "Mateo", "Natalia", "Oscar", "Paola", "Quique", "Rosa", "Santiago", "Ana"]
apellidos = ["Bastidas", "Perez", "Gomez", "Lopez", "Torres", "Ruiz", "Castro", "Mora", "Vera", "Rojas", 
             "Silva", "Acosta", "Paz", "Cruz", "Sosa", "Vega", "Rios", "Luna", "Soto", "Ramos"]
telefonos = [f"09872080{i:02d}" for i in range(20)]
descripciones = ["Documentos", "Electronica", "Ropa", "Libros", "Zapatos", "Juguetes", "Comida", "Herramientas", "Perfume", "Reloj", 
                 "Maquillaje", "Accesorios", "Medicinas", "Vidrio", "Cuadernos", "Bolso", "Celular", "Cargador", "Llaves", "Es una Bomba"]
codigos_postales = ["170110", "170150", "170200", "170300", "170400", "170500", "170120", "170180"]
nombres_zonas = ["Sur", "Norte", "Centro", "Valle", "Oeste", "Este", "Sur-Oeste", "Norte-Este"]

for i in range(20):
    sala.enqueue(Clientes(nombres[i], apellidos[i], telefonos[i], i))

for i in range(20):
    paqueteria.llamar_siguiente(sala, i * 5)
    codigo_unico = generar_codigo(tabla_hash)
    cp = codigos_postales[i % len(codigos_postales)]
    nz = nombres_zonas[i % len(nombres_zonas)]
    nueva_zona = Zona(cp, nz)

    zona_encontrada, _ = busqueda_binaria_zona(lista_zonas, nueva_zona.codigo_postal)
    if zona_encontrada is None:
        lista_zonas.append(nueva_zona)
        lista_zonas.sort()
        zona_a_usar = nueva_zona
    else:
        zona_a_usar = zona_encontrada

    nombre_completo = f"{nombres[i]} {apellidos[i]}"
    paquetenuevo = Paquete(codigo_unico, nombre_completo, zona_a_usar, descripciones[i])
    paquetes.enqueue(paquetenuevo)
    tabla_hash.put(codigo_unico, {"cliente": nombre_completo, "zona": zona_a_usar, "descripcion": descripciones[i]})
    paqueteria.terminar((i * 5) + 3)

print("--- PRUEBAS DE CASOS LÍMITE ---")
sala_vacia = Cola()
print("Atender cola vacía:", sala_vacia.dequeue())
res, comps = tabla_hash.buscar_hash("TRK-999")
print(f"Buscar código inexistente 'TRK-999' -> Resultado: {res}, Comparaciones: {comps}")
print("Deshacer sin historial:", historial.deshacer())

### Mínimo 8 · Medición de comparaciones
Midan cuántas comparaciones hace cada búsqueda (secuencial, binaria y hash) con **100, 1000 y 10000** datos y muestren una tabla con los resultados. Después escriban en la celda siguiente qué observan.

In [ ]:
print("--- MEDICIÓN DE RENDIMIENTO ---")
print(f"{'Escala (N)':<12} | {'Secuencial (Comps)':<20} | {'Binaria (Comps)':<16} | {'Hash (Comps)':<14}")
print("-" * 68)

for escala in [100, 1000, 10000]:
    t_hash = HashTable(escala * 2)
    l_zonas = [Zona(str(170000 + i), f"Zona_{i}") for i in range(escala)]
    l_zonas.sort()
    l_paquetes = [Paquete(f"TRK-{i}", f"Cliente_{i}", l_zonas[i % len(l_zonas)]) for i in range(escala)]
    
    for p in l_paquetes:
        t_hash.put(p.codigo_seguimiento, p)

    _, comps_seq = seq_search_todos(l_paquetes, f"Cliente_{escala-1}")
    _, comps_bin = busqueda_binaria_zona(l_zonas, l_zonas[-1].codigo_postal)
    _, comps_hash = t_hash.buscar_hash(l_paquetes[-1].codigo_seguimiento)

    print(f"{escala:<12} | {comps_seq:<20} | {comps_bin:<16} | {comps_hash:<14}")

### Análisis de las mediciones
¿Qué pasa con las comparaciones al multiplicar los datos por 10? ¿Cuál búsqueda crece más rápido? ¿Su tabla hash se comportó como esperaban?

*(escriban aquí su análisis)*

### Lista de verificación antes de entregar
- [ ] Los 8 mínimos funcionan y están demostrados
- [ ] La tabla hash no usa `dict` ni `set` y maneja colisiones
- [ ] El código está comentado y ordenado
- [ ] Los 5 integrantes pueden explicar **cualquier** parte, no solo la suya
